In [ ]:
import os
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reproduce.py").is_file())
os.chdir(ROOT)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
  
PROJECT_PATH = Path("data/genomics")

In [ ]:
meta = pd.read_csv(PROJECT_PATH / "reference/metadata_complete.csv")
meta['population'] = meta['Strain'] + '_' + meta['Culture'].astype(str).str.zfill(2)
meta

In [ ]:
dfs = []

for i, row in meta.iterrows():
    df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
    df["Strain"] = row['Strain']
    df["Culture"] = row['Culture']
    df["Day"] = int(row['Day'])
    dfs.append(df)

df = pd.concat(dfs)
df


In [ ]:
select_lineage = "PA"
pops = meta.query(f'Strain=="{select_lineage}"')['population'].unique()
timepoints = sorted(meta.query(f'Strain=="{select_lineage}"')['Day'].unique())
print(pops)
print(timepoints)

In [ ]:
meta.query(f'population=="{select_lineage}_01"')

In [ ]:
def traceAlleleFreq(pop, min_freq=0.33):

    D = []
    sorted_meta = meta.query(f'population=="{pop}"').sort_values(by='Day')
    for i, row in sorted_meta.iterrows():
        df = pd.read_csv(PROJECT_PATH / f"out/{row['FolderDate']}/{row['source_file']}/output/output.gd.tsv", sep='\t')
        D.append(df)

    # Remove mutations detected in the wild-type background
    # 1. Select background mutations with strong signal
    wt_background=D[0].loc[D[0].frequency>0.01, 'position']
    #print(f"WT background mutations: {wt_background.values}")
    D_=[]

    #print(f"# of mutations\t# of (freq>{min_freq}):")
    for i in range(len(D)):
        # 2. Filter out mutations by position on the chromosome
        df=D[i][~D[i]['position'].isin(wt_background)]
        D_.append(df)
        #print(f"{df.shape[0]}\t{sum(df['frequency']>min_freq)}")
    
    tracked_positions=[]
    for i in range(len(D_)):
        d=D_[i]
        tracked_positions.extend(list(d.loc[ (d['frequency']>min_freq), 'position' ])) #& ~d['mutation_category'].isin(['mobile_element_insertion','large_deletion']), 'position' ] ))
    tracked_positions=set(tracked_positions)
    print(f"Population {pop} # of tracked mutations: {len(tracked_positions)}")

    ## 3. Trace the frequency of those mutations
    T=[]
    for pos in tracked_positions:

        freq=[]
        for i in range(len(D_)):
            d = D_[i]
            ind = d['position']==pos
            
            if sum(ind)<1:
                freq.append(0)
            else:
                freq.append( (d.loc[ind , 'frequency'].values)[0] )
                gene_name = d.loc[ind, 'gene_name'].values[0]
                gene_product = d.loc[ind, 'gene_product'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                aa_new_seq = d.loc[ind, 'aa_new_seq'].values[0]
                aa_ref_seq = d.loc[ind, 'aa_ref_seq'].values[0]
                new_seq = d.loc[ind, 'new_seq'].values[0]
                codon_ref_seq = d.loc[ind, 'codon_ref_seq'].values[0]
                aa_pos = d.loc[ind, 'aa_position'].values[0]
                gene_pos = d.loc[ind, 'gene_position'].values[0]
                mut_cat = d.loc[ind, 'mutation_category'].values[0]

        T.append({'position': pos, 'freq': np.round(freq,2), 
                'gene_name':gene_name, 'gene_product':gene_product,
                'aa_ref_seq':aa_ref_seq, 'aa_new_seq':aa_new_seq,
                'aa_pos': aa_pos, 'gene_pos':gene_pos, 'mut_cat': mut_cat,
                'new_seq': new_seq, 'codon_ref_seq': codon_ref_seq})

    T=pd.DataFrame(T)
    T.fillna({'aa_ref_seq': '',
              'aa_new_seq': '',
              'aa_pos': ''},inplace=True)
    
    nsi = T['aa_pos']==''
    T.loc[nsi,'label'] = T.loc[nsi,'gene_name'] + ' ' + T.loc[nsi, 'mut_cat']
    T.loc[~nsi, 'label'] = T.loc[~nsi,'gene_name'] + ' ' + T.loc[~nsi,'aa_ref_seq'] + T.loc[~nsi, 'aa_pos'].astype(str).str.replace(r'\.0','',regex=True) + T.loc[~nsi,'aa_new_seq']
        # T.loc[nsi,'gene_pos'] + ' '
    T.sort_values(by=['mut_cat','gene_name'],ascending=False,inplace=True)
    T.reset_index(inplace=True,drop=True)
    
    return T

In [ ]:
af=[]
for pop in pops:
    # print(traceAlleleFreq(pop, min_freq=0.2).shape)
    af.append(traceAlleleFreq(pop, min_freq=0.1))
    af[-1].to_csv(f"data/genomics/data/processed/traced_alleles/{select_lineage}/{pop}.csv", index=False)

In [ ]:
pd.set_option("display.max_rows", 100)
af[3].head(100)

In [ ]:


# Assuming af is a list of DataFrames
compiled_unique_labels = set()  # Use a set to avoid duplicates

# Loop through the list of DataFrames
for i in range(10):  # Iterate over af[0] to af[9]
    df = af[i]
    
    if isinstance(df, pd.DataFrame):
        # Filter for rows where 'gene_name' is 'fusA'
        filtered_df = df[df['gene_name'] == 'trkH']
        
        # Get the unique labels and update the set
        unique_labels = filtered_df['label'].unique()
        compiled_unique_labels.update(unique_labels)
    else:
        print(f"Error: af[{i}] is not a DataFrame.")

# Convert the set to a sorted list (optional)
compiled_unique_labels_list = sorted(list(compiled_unique_labels))

print(compiled_unique_labels_list)




In [ ]:
all = []
for ix, df in enumerate(af):
    dff = df.copy()
    dff['Pop'] = ix+1 
    all.append(dff)
combined_df=pd.concat(all)


combined_df['last_freq'] = combined_df['freq'].apply(lambda x: x[-1])
combined_df


In [ ]:
def get_mutation_group_and_labels(group):
    # """
    # Returns select mutations and alternate labels based on the specified group.
    
    # Parameters:
    #     group (str): The group to retrieve mutations for. Options are 'prs_phoQ' or 'other'.
    
    # Returns:
    #     tuple: A tuple containing a list of selected mutations and a dictionary of alternate labels.
    # """
    # Select mutations based on group
    if group == 'fusA_trkH':
        select_mutations = [
            'fusA P610L', 'trkH L80Q', 'trkH T20P', 'trkH G156C',
            'trkH V155E', 'trkH small_indel',  'trkH S105Y', 'trkH Q159H', 'trkH L185Q'
        ]
        alt_labels = {
            'trkH small_indel': 'trkH indel',
        }
        manual_label_order = [
            'fusA P610L', 'trkH L80Q', 'trkH T20P', 'trkH G156C', 
            'trkH V155E', 'trkH small_indel',  'trkH S105Y', 'trkH Q159H', 'trkH L185Q'
]
    elif group == 'other':
        select_mutations = [
            'pgsA V44E', 'bluF I39L', 'uvrC V593E', 'osmY L198Q', 'xerC V56V', 
            'glpF V52E', 'tufB D110Y', 'yaaA I52T', 'yacH A495T','gspD D55A',
            'metL S631T', 'baeS F3L','ybiO/glnQ snp_intergenic', 'trkG G159D',
            'yfjW/yfjX snp_intergenic', 'sapF R158S', 'fdrA I220T'
        ]
        alt_labels = {
            'ybiO/glnQ snp_intergenic': 'ybiO/glnQ snp intergenic',
            'yfjW/yfjX snp_intergenic': 'yfjW/yfjX snp intergenic',

        }
        manual_label_order = [
            'pgsA V44E', 'bluF I39L', 'uvrC V593E', 'osmY L198Q', 'xerC V56V', 
            'glpF V52E', 'tufB D110Y', 'yaaA I52T', 'yacH A495T','gspD D55A',
            'metL S631T', 'baeS F3L','ybiO/glnQ snp_intergenic', 'trkG G159D',
            'yfjW/yfjX snp_intergenic', 'sapF R158S', 'fdrA I220T'
]
    else:
        raise ValueError("Invalid group. Options are 'fusA_trkH' or 'other'.")

    return select_mutations, alt_labels, manual_label_order


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"

# Assuming `af` is a list of DataFrames with the following structure
# Example structure for af: [{'label': '...', 'gene_name': '...', 'freq': [..]}]
# af = [...]
standard_map = plt.cm.get_cmap('Blues')

# Create a new colormap that interpolates between white and the 'Greens' colormap
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # Replace the first color (for 0 values) with white
custom_map = mcolors.ListedColormap(new_colors)

af
# Define the gene names to include (set to None or empty list for no filtering)
designated_gene_names = ['fusA', 'trkH']  # Example gene names to include; set to [] for no filtering

# Extract unique labels and their counts from the data
all_labels = np.concatenate([df['label'].unique() for df in af])
unique_labels, counts = np.unique(all_labels, return_counts=True)

# Create a DataFrame with labels and counts
label_counts = pd.DataFrame({'label': unique_labels, 'count': counts})

# Map gene_name and the last freq value to labels
gene_name_map = {row['label']: row['gene_name'] for df in af for _, row in df.iterrows()}
freq_map = {row['label']: row['freq'][-1] for df in af for _, row in df.iterrows()}  # Assuming 'freq' is a list or array
label_counts['gene_name'] = label_counts['label'].map(gene_name_map)
label_counts['freq'] = label_counts['label'].map(freq_map)

# Define a mapping of old labels to new alternate labels
label_mapping = {
    'mrcB|mrcB T|T702|657S|S': 'mrcB T702S',
    'mgtL/mgtA small_indel': 'mgtL/mgtA indel',
    'trkH small_indel': 'trkH indel',
    'yicC mobile_element_insertion': 'yicC IS2 insertion'
}

# Apply the label mapping to the label_counts DataFrame
label_counts['label'] = label_counts['label'].replace(label_mapping)

# Apply the label mapping to combined_df to match the updated labels
combined_df['label'] = combined_df['label'].replace(label_mapping)

# Optional: Filter `label_counts` and `combined_df` if designated_gene_names is not empty
if designated_gene_names:
    label_counts = label_counts[label_counts['gene_name'].isin(designated_gene_names)]
    combined_df = combined_df[combined_df['gene_name'].isin(designated_gene_names)]

# Separate labels by gene_name
fusA_labels = label_counts[label_counts['gene_name'] == 'fusA']
trkH_labels = label_counts[label_counts['gene_name'] == 'trkH']
other_labels = label_counts[~label_counts['gene_name'].isin(['fusA', 'trkH'])]

# Sort `fusA` and `trkH` groups by frequency
fusA_labels = fusA_labels.sort_values(by='freq', ascending=True)
trkH_labels = trkH_labels.sort_values(by='freq', ascending=False)

# Sort other labels by last freq value (descending)
other_labels = other_labels.sort_values(by='freq', ascending=False)

# Combine the groups: fusA first, trkH second, others last
label_counts_sorted = pd.concat([fusA_labels, trkH_labels, other_labels], ignore_index=True)

# Create a categorical order for labels
label_order = label_counts_sorted['label'].tolist()

# Filter the main DataFrame to include only the sorted labels
filtered_df = combined_df[combined_df['label'].isin(label_order)]

# Pivot the DataFrame to have 'label' as rows, 'Pop' as columns, and 'last_freq' as values
heatmap_data = filtered_df.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# Reindex the heatmap data to match the label order
heatmap_data = heatmap_data.reindex(label_order)

# Convert the DataFrame to a numpy array for sorting
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns

cell_width = .7  # Width of each cell

# Calculate figure size
figsize = ((9*cell_width), 6)

# Create a figure and axes with the calculated size
fig, ax = plt.subplots(figsize=figsize)
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# Create the heatmap with custom settings, including inverted colors
sns.heatmap(freq_mat.T, yticklabels=timepoints, xticklabels=False, cmap=custom_map, vmin=0, vmax=1, ax=ax,
            annot=annot_matrix, fmt="", annot_kws={'size': 20, 'ha': 'center', 'va': 'center'}, cbar=False,
            linewidths=.5,  # Thickness of the grid lines
            linecolor='lightgrey'  # Color of the grid lines
)

# Set labels and title with larger font sizes
ax.set_ylabel('Culture Number', fontsize=25)  # Set y-axis label with larger font
ax.set_xlabel('', fontsize=25)  # Set x-axis label with larger font
#ax.set_title('High Frequency Mutations of MG$^{{\\mathrm{{AMI}}}}$', fontsize=25)  # Set title with larger font

# Adjust tick label font sizes
ax.tick_params(axis='both', which='major', labelsize=15, length=0)

# Rotate x-axis labels for readability with larger font sizes
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)  # Make spines visible
    spine.set_linewidth(2)   # Set the border width
    spine.set_color("black") # Set the border color

# Display the plot
plt.tight_layout()
plt.show()

# Save the figure

fig.savefig(PROJECT_PATH / "figures/final/PA_fusA+trkH_sorted_no_labels.png", dpi=600, bbox_inches='tight')


In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
plt.rcParams["font.family"] = "Nimbus Roman"



# --- Define alt_labels and manual_label_order manually (no group distinction) ---
alt_labels = {
    'fusA P610L': 'FusA P610L',
    'trkH L80Q': 'TrkH L80Q',
    'trkH T20P': 'TrkH T20P',
    'trkH G156C': 'TrkH G156C',
    'trkH V155E': 'TrkH V155E',
    'trkH small_indel': r'$\it{trkH}$ indel',
    'trkH S105Y': 'TrkH S105Y',
    'trkH Q159H': 'TrkH Q159H',
    'trkH L185Q': 'TrkH L185Q',
    'pgsA V44E': 'PgsA V44E',
    'bluF I39L': 'BluF I39L',
    'uvrC V593E': 'UvrC V593E',
    'osmY L198Q': 'OsmY L198Q',
    'xerC V56V': 'XerC V56V',
    'glpF V52E': 'GlpF V52E',
    'tufB D110Y': 'TufB D110Y',
    'yaaA I52T': 'YaaA I52T',
    'yacH A495T': 'YacH A495T',
    'gspD D55A': 'GspD D55A',
    'metL S631T': 'MetL S631T',
    'baeS F3L': 'BaeS F3L',
    'ybiO/glnQ snp_intergenic': r'$\it{ybiO/glnQ}$ snp intergenic',
    'trkG G159D': 'TrkG G159D',
    'yfjW/yfjX snp_intergenic': r'$\it{yfjW/yfjX}$ snp intergenic',
    'sapF R158S': 'SapF R158S',
    'fdrA I220T': 'FdrA I220T'
}

manual_label_order = [
    'fusA P610L', 'trkH L80Q', 'trkH T20P', 'trkH G156C', 'trkH V155E', 'trkH small_indel',
    'trkH S105Y', 'trkH Q159H', 'trkH L185Q',
    'pgsA V44E', 'bluF I39L', 'uvrC V593E', 'osmY L198Q', 'xerC V56V',
    'glpF V52E', 'tufB D110Y', 'yaaA I52T', 'yacH A495T', 'gspD D55A',
    'metL S631T', 'baeS F3L', 'ybiO/glnQ snp_intergenic', 'trkG G159D',
    'yfjW/yfjX snp_intergenic', 'sapF R158S', 'fdrA I220T'
]

# --- Colormap ---
standard_map = plt.cm.get_cmap('Blues')
new_colors = standard_map(np.linspace(0, 1, 256))
new_colors[0] = np.array([1, 1, 1, 1])  # 0 = white
custom_map = mcolors.ListedColormap(new_colors)

# --- Add 'Pop' column if needed ---
for idx, df in enumerate(af):
    if 'Pop' not in df.columns:
        df['Pop'] = idx + 1

# --- Create freq map ---
freq_map = {
    (row['label'], row['Pop']): row['freq'][-1] if isinstance(row['freq'], (list, np.ndarray)) else row['freq']
    for df in af for _, row in df.iterrows()
}

# --- Add last_freq ---
for df in af:
    df['last_freq'] = df.apply(lambda row: freq_map.get((row['label'], row['Pop']), 0), axis=1)

# --- Combine all data ---
all_data = pd.concat(af, ignore_index=True)

# --- Pivot table ---
heatmap_data = all_data.pivot_table(index='label', columns='Pop', values='last_freq', fill_value=0)

# --- Ensure all culture columns ---
all_cultures = range(1, 11)
heatmap_data = heatmap_data.reindex(columns=all_cultures, fill_value=0)

# --- Sort rows: manual first, then the rest ---
present_labels = heatmap_data.index.tolist()
present_manual_labels = [label for label in manual_label_order if label in present_labels]
remaining_labels = [label for label in present_labels if label not in manual_label_order]
final_row_order = present_manual_labels + remaining_labels
heatmap_data = heatmap_data.reindex(index=final_row_order)

# --- Plot matrix and labels ---
freq_mat = heatmap_data.values
timepoints = heatmap_data.columns
plot_labels = [alt_labels.get(label, label) for label in heatmap_data.index]
annot_matrix = np.where(freq_mat.T == 0, '', freq_mat.T.round(1).astype(str))

# --- Figure size ---
cell_width = 0.7
figsize = ((heatmap_data.shape[0] * cell_width), 7)

# --- Plot heatmap ---
fig, ax = plt.subplots(figsize=figsize)
sns.heatmap(
    freq_mat.T,
    yticklabels=timepoints,
    xticklabels=plot_labels,
    cmap=custom_map,
    vmin=0,
    vmax=1,
    ax=ax,
    annot=annot_matrix,
    fmt="",
    annot_kws={'size': 20, 'ha': 'center', 'va': 'center'},
    cbar=False,
    linewidths=0.5,
    linecolor='lightgrey'
)

# --- Style ---
ax.set_ylabel('Culture Number', fontsize=25)
ax.set_xlabel('', fontsize=25)
ax.tick_params(axis='both', which='major', labelsize=15, length=0)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=20)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=20)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(2)
    spine.set_color("black")

plt.subplots_adjust(bottom=0.1)
plt.tight_layout()

# --- Save ---
fig.savefig(PROJECT_PATH / "figures/final/PA_full.png", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.font_manager as fm

# Get a list of all available font names
font_names = sorted({f.name for f in fm.fontManager.ttflist})
for name in font_names:
    print(name)